In [ ]:

# Install and import libraries
!pip -q install kagglehub

import kagglehub
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Kaggle dataset
path = kagglehub.dataset_download("sajid576/sql-injection-dataset")

# Read CSV file
csv_file = next(
    os.path.join(root, f)
    for root, dirs, files in os.walk(path)
    for f in files if f.endswith(".csv")
)
df = pd.read_csv(csv_file)

print("Dataset loaded:", df.shape)

# Identify query and label columns
label_col = next(c for c in df.columns
                 if c.lower() in ["label", "class", "category"])

query_col = next(c for c in df.columns
                 if c != label_col and df[c].dtype == "object")

df = df[[query_col, label_col]].dropna()

# Convert queries into numerical features

vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df[query_col].astype(str))
y = df[label_col]

# Encode labels
if not pd.api.types.is_numeric_dtype(y):
    y = y.astype(str).str.lower().map(
        lambda x: 0 if x in ["0", "normal", "benign", "safe"] else 1
    )

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Decision Tree
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
predictions = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, predictions) * 100, "%")
print("\nClassification Report:")
print(classification_report(y_test, predictions))


# Test a SQL query
query = ["SELECT * FROM users WHERE id = 1 OR 1=1"]

result = model.predict(vectorizer.transform(query))

print("Query Classification:",
      "Malicious" if result[0] == 1 else "Normal")


Using Colab cache for faster access to the 'sql-injection-dataset' dataset.
Dataset loaded: (30919, 2)

Accuracy: 99.40168175937904 %

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3908
           1       1.00      0.99      0.99      2276

    accuracy                           0.99      6184
   macro avg       0.99      0.99      0.99      6184
weighted avg       0.99      0.99      0.99      6184

Query Classification: Malicious
